# LIBRARY

In [8]:
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, make_scorer, roc_auc_score
import shap

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

import statsmodels.api as sm

# LOAD

In [9]:
with open("PROCESSED/DATA/merged_and_dropped.cat_cols.json") as f:
    cat_cols = json.load(f)

X_train = pd.read_parquet("INPUTS/TRAIN/X_train.parquet")
X_test = pd.read_parquet("INPUTS/TEST/X_test.parquet")
y_train = pd.read_parquet("INPUTS/TRAIN/y_train.parquet")
y_test = pd.read_parquet("INPUTS/TEST/y_test.parquet")

X_train[cat_cols] = X_train[cat_cols].astype("category")
X_test[cat_cols] = X_test[cat_cols].astype("category")

# one-hot encode categorical variables
X_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

# ensure target is categorical
y_train_cat = y_train.iloc[:, 0].astype("category")
y_test_cat  = y_test.iloc[:, 0].astype("category")

# BUILD PIPELINE
- `optimize_model`: Utilizes GridSearchCV for hyperparameter optimization
- `evaluate_best_model`: Evaluates the best model on train and test data
- `save_results`: Saves results and computes and saves SHAP values

In [10]:
# CONFIGURATION
cv_random_state = 42
base_path = Path("RESULTS/BASELINES")
baseline_probs_path = base_path / "PROBABILITIES"
baseline_SHAP_path = base_path / "SHAP"
baseline_performance_path = base_path / "PERFORMANCE"
baseline_params_path = base_path / "PARAMETERS"

def optimize_model(model, X, y, param_grid, verbose = True, save_results = True):
    scorers = {
        'auc': 'roc_auc_ovr',
        'accuracy': make_scorer(accuracy_score),
        'f1': make_scorer(f1_score, average='macro')
    }
    refit = 'f1'

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=cv_random_state)

    grid = GridSearchCV(
        estimator = model,
        param_grid = param_grid,
        scoring = scorers,
        refit = refit,
        cv = cv,
        n_jobs = -1,
        verbose = 1
    )
    grid.fit(X, y)
    
    if verbose:
        print("Best parameters:", grid.best_params_)
        print(f"Best {refit} CV score:", grid.best_score_)
        # print("CV AUC at best-AUC params:", grid.cv_results_['mean_test_auc'][grid.best_index_])
        # print("CV F1 at best-AUC params:", grid.cv_results_['mean_test_f1'][grid.best_index_])
        # print("CV Accuracy at best-AUC params:", grid.cv_results_['mean_test_accuracy'][grid.best_index_])
        idx = grid.best_index_
        print(f"CV    accuracy : {grid.cv_results_['mean_test_accuracy'][idx]:.3f}, "
            f"F1: {grid.cv_results_['mean_test_f1'][idx]:.3f}, "
            f"AUC: {grid.cv_results_['mean_test_auc'][idx]:.3f}")

    return grid

def evaluate_best_model(model, X_train, y_train, X_test, y_test, save_results = True):
    best_model = model.best_estimator_

    y_pred_train = best_model.predict(X_train)
    y_pred_test  = best_model.predict(X_test)

    acc_train = accuracy_score(y_train, y_pred_train)
    acc_test  = accuracy_score(y_test, y_pred_test)

    f1_train = f1_score(y_train, y_pred_train, average='macro')
    f1_test  = f1_score(y_test, y_pred_test, average='macro')

    auc_train = roc_auc_score(y_train, best_model.predict_proba(X_train)[:, 1])
    auc_test  = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

    print(f"Train accuracy : {acc_train:.3f}, F1: {f1_train:.3f}, AUC: {auc_train:.3f}")
    print(f"Test  accuracy : {acc_test:.3f}, F1: {f1_test:.3f}, AUC: {auc_test:.3f}")

    print("\nClassification report:\n")
    cls_report = classification_report(y_test, y_pred_test)
    print(cls_report)
    return {
        'acc_train': acc_train, 
        'acc_test': acc_test,
        'f1_train': f1_train,
        'f1_test': f1_test,
        'auc_train': auc_train,
        'auc_test': auc_test,
        'cls_report': cls_report
    }

# Saves results + computes and saves SHAP values
def save_results(model, X_train, y_train, X_test, y_test, performance_dict):
    best_model = model.best_estimator_
    best_params = model.best_params_
    cv_results = model.cv_results_
    best_index = model.best_index_

    model_name = best_model.__class__.__name__
    os.makedirs(baseline_probs_path, exist_ok=True)
    os.makedirs(baseline_SHAP_path, exist_ok=True)
    os.makedirs(baseline_performance_path, exist_ok=True)
    os.makedirs(baseline_params_path, exist_ok=True)

    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_train) # Compute shap values

    # SAVE PARAMETERS
    with open(baseline_params_path / f"{model_name}_BEST_HYPER.json", "w") as f:
        json.dump(best_params, f)

    # SAVE PROBABILITY OUTPUTS
    test_df = pd.DataFrame(best_model.predict_proba(X_test))
    test_df["y_true"] = np.asarray(y_test)
    test_df.to_csv(baseline_probs_path / f"{model_name}_PROBS.csv", index=False)

    # SAVE SHAP IMPORTANCES
    if shap_values.ndim == 3: # "Multiclass" case (some models treat binary as multiclass)
        mean_abs_shap_class = np.abs(shap_values).mean(axis=(0,2))
    else: # binary case
        mean_abs_shap_class = np.abs(shap_values).mean(axis=0)
    shap_df = pd.DataFrame(mean_abs_shap_class, index=X_train.columns, columns=["mean_abs_shap"])
    shap_df.to_csv(baseline_SHAP_path / f"{model_name}_SHAP.csv", index=True)
    
    # SAVE PERFORMANCE
    performance = {
        "model": model_name,
        "acc_train": performance_dict['acc_train'],
        "acc_test": performance_dict['acc_test'],
        "f1_train": performance_dict['f1_train'],
        "f1_test": performance_dict['f1_test'],
        "auc_train": performance_dict['auc_train'],
        "auc_test": performance_dict['auc_test'],
        "best_cv_auc": cv_results['mean_test_auc'][best_index],
        "best_cv_f1": cv_results['mean_test_f1'][best_index],
        "best_cv_accuracy": cv_results['mean_test_accuracy'][best_index]
    }
    performance = pd.DataFrame([performance])
    performance.to_csv(baseline_performance_path / f"{model_name}_PERF.csv", index=False)
    
    with open(baseline_performance_path / f"{model_name}_CLASSIFICATION_REPORT.txt", "w") as f:
        f.write(performance_dict['cls_report'])

# MODELS

### RANDOM FOREST

In [11]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

model = optimize_model(rf, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best f1 CV score: 0.598058884647802
CV    accuracy : 0.850, F1: 0.598, AUC: 0.875
Train accuracy : 0.935, F1: 0.866, AUC: 0.995
Test  accuracy : 0.871, F1: 0.639, AUC: 0.895

Classification report:

              precision    recall  f1-score   support

         0.0       0.87      0.99      0.93      1642
         1.0       0.83      0.22      0.35       306

    accuracy                           0.87      1948
   macro avg       0.85      0.61      0.64      1948
weighted avg       0.87      0.87      0.84      1948



### XGB

In [12]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

model = optimize_model(xgb, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Best f1 CV score: 0.749541889820258
CV    accuracy : 0.876, F1: 0.750, AUC: 0.901
Train accuracy : 0.939, F1: 0.883, AUC: 0.975
Test  accuracy : 0.899, F1: 0.785, AUC: 0.920

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      1642
         1.0       0.74      0.54      0.63       306

    accuracy                           0.90      1948
   macro avg       0.83      0.75      0.78      1948
weighted avg       0.89      0.90      0.89      1948



### EXTRA TREES

In [13]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

et = ExtraTreesClassifier(random_state=42, n_jobs=-1)

model = optimize_model(et, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Best f1 CV score: 0.5280642566362473
CV    accuracy : 0.838, F1: 0.528, AUC: 0.856
Train accuracy : 0.906, F1: 0.787, AUC: 0.982
Test  accuracy : 0.856, F1: 0.556, AUC: 0.861

Classification report:

              precision    recall  f1-score   support

         0.0       0.86      1.00      0.92      1642
         1.0       0.80      0.11      0.19       306

    accuracy                           0.86      1948
   macro avg       0.83      0.55      0.56      1948
weighted avg       0.85      0.86      0.81      1948



### CATBOOST

In [14]:
param_grid = {
    'n_estimators': [200, 400],
    'depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
}

cat = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    thread_count=-1,
    verbose=0  # silence per-iteration output
)

model = optimize_model(cat, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best parameters: {'depth': 4, 'learning_rate': 0.1, 'n_estimators': 400}
Best f1 CV score: 0.7487702225191676
CV    accuracy : 0.876, F1: 0.749, AUC: 0.898
Train accuracy : 0.964, F1: 0.932, AUC: 0.988
Test  accuracy : 0.892, F1: 0.768, AUC: 0.911

Classification report:

              precision    recall  f1-score   support

         0.0       0.91      0.96      0.94      1642
         1.0       0.72      0.51      0.60       306

    accuracy                           0.89      1948
   macro avg       0.82      0.74      0.77      1948
weighted avg       0.88      0.89      0.88      1948



### HIST GRADIENT BOOSTING

In [15]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'max_iter': [100, 200],
    'min_samples_leaf': [20, 50],
}

hgb = HistGradientBoostingClassifier(
    random_state=42
)

model = optimize_model(hgb, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'max_iter': 200, 'min_samples_leaf': 50}
Best f1 CV score: 0.7504319447204221
CV    accuracy : 0.876, F1: 0.750, AUC: 0.894
Train accuracy : 0.994, F1: 0.990, AUC: 1.000
Test  accuracy : 0.899, F1: 0.789, AUC: 0.916

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.96      0.94      1642
         1.0       0.73      0.57      0.64       306

    accuracy                           0.90      1948
   macro avg       0.83      0.76      0.79      1948
weighted avg       0.89      0.90      0.89      1948



### LIGHTGBM

In [16]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10],
    'learning_rate': [0.05, 0.1],
    'num_leaves': [31, 63],
}

lgbm = LGBMClassifier(
    objective='binary',
    random_state=42,
    n_jobs=-1,
    verbose=-1      # <- this suppresses the info messages
)

model = optimize_model(lgbm, X_encoded, y_train_cat, param_grid)
performance_dict = evaluate_best_model(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat)
save_results(model, X_encoded, y_train_cat, X_test_encoded, y_test_cat, performance_dict)

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'num_leaves': 63}
Best f1 CV score: 0.7479702922817907
CV    accuracy : 0.877, F1: 0.748, AUC: 0.897
Train accuracy : 0.998, F1: 0.996, AUC: 1.000
Test  accuracy : 0.900, F1: 0.789, AUC: 0.916

Classification report:

              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      1642
         1.0       0.75      0.55      0.64       306

    accuracy                           0.90      1948
   macro avg       0.83      0.76      0.79      1948
weighted avg       0.89      0.90      0.89      1948



c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:583: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


### UNTUNED GLM

Special Handling for the baseline GLM model, since it uses statsmodels instead of sklearn + is not a tree model

In [17]:
X_train_glm_untuned = sm.add_constant(X_encoded.astype(float))
X_test_glm_untuned = sm.add_constant(X_test_encoded.astype(float))

glm_untuned = sm.GLM(
    y_train,
    X_train_glm_untuned,
    family = sm.families.Binomial()
).fit()

# Predictions
y_pred_train_glm_untuned = glm_untuned.predict(X_train_glm_untuned)
y_pred_test_glm_untuned = glm_untuned.predict(X_test_glm_untuned)

y_pred_train_cat_untuned = (y_pred_train_glm_untuned >= 0.5).astype(int)
y_pred_test_cat_untuned = (y_pred_test_glm_untuned >= 0.5).astype(int)
acc_train_untuned = accuracy_score(y_train_cat, y_pred_train_cat_untuned)
acc_test_untuned = accuracy_score(y_test_cat, y_pred_test_cat_untuned)
f1_train_untuned = f1_score(y_train_cat, y_pred_train_cat_untuned, average='macro')
f1_test_untuned = f1_score(y_test_cat, y_pred_test_cat_untuned, average='macro')
auc_train_untuned = roc_auc_score(y_train_cat, y_pred_train_glm_untuned)
auc_test_untuned = roc_auc_score(y_test_cat, y_pred_test_glm_untuned)

print(f"Train accuracy: {acc_train_untuned:.3f},  F1: {f1_train_untuned:.3f}, AUC: {auc_train_untuned:.3f}")
print(f"Test  accuracy: {acc_test_untuned:.3f},  F1: {f1_test_untuned:.3f}, AUC: {auc_test_untuned:.3f}")
print("\nClassification report:\n")
cls_report = classification_report(y_test_cat, y_pred_test_cat_untuned)
print(cls_report)

# Probabilty Outputs
test_df_untuned = pd.DataFrame(y_pred_test_glm_untuned, columns=["prob_1"])
test_df_untuned["y_true"] = np.asarray(y_test)
test_df_untuned.to_csv(baseline_probs_path / "GLM_UNTUNED_PROBS.csv", index=False)

# Performance metrics
model_name = "GLM_UNTUNED"
perf_row = {
    "model": model_name,
    "acc_train": acc_train_untuned,
    "acc_test": acc_test_untuned,
    "f1_train": f1_train_untuned,
    "f1_test": f1_test_untuned,
    "auc_train": auc_train_untuned,
    "auc_test": auc_test_untuned,
}
perf_df = pd.DataFrame([perf_row])
perf_df.to_csv(baseline_performance_path / f"{model_name}_PERF.csv", index=False)

with open(baseline_performance_path / f"{model_name}_CLASSIFICATION_REPORT.txt", "w") as f:
    f.write(cls_report)

print(glm_untuned.summary())

c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


Train accuracy: 0.835,  F1: 0.768, AUC: 0.844
Test  accuracy: 0.828,  F1: 0.741, AUC: 0.815

Classification report:

              precision    recall  f1-score   support

         0.0       0.96      0.83      0.89      1642
         1.0       0.47      0.80      0.59       306

    accuracy                           0.83      1948
   macro avg       0.71      0.82      0.74      1948
weighted avg       0.88      0.83      0.84      1948



c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)
c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: divide by zero encountered in log
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +
c:\Users\victo\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\genmod\families\family.py:1056: RuntimeWarning: invalid value encountered in multiply
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +


                 Generalized Linear Model Regression Results                  
Dep. Variable:            IS_DIABETES   No. Observations:                 7789
Model:                            GLM   Df Residuals:                     7395
Model Family:                Binomial   Df Model:                          393
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                    nan
Date:                Thu, 27 Nov 2025   Deviance:                   1.1808e+05
Time:                        12:36:30   Pearson chi2:                 5.77e+18
No. Iterations:                   100   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------